# Operative-note generation demo (Colab)

Facts (PitVis labels or CountGD+SAM2 detections) -> Qwen draft note ->
assembled note, using the `notegen` package in `code/models/qwen3vl_finetune/src/`
and the `notegen_grounding`/`note_format` GRPO reward ORMs in
`code/rlvr/notegen_plugin.py` (usable standalone here, no ms-swift needed for
this demo -- only for the actual GRPO training run, `train_grpo.effective.sh`).

**Read `README.md` and `notes.md` in this experiment folder first** -- in
particular: there is no gold operative-note text corpus (the reward is
grounding-based, not reference-comparison). Model: base `Qwen/Qwen3.5-9B` +
the `A_s189` LoRA adapter (a counting/VQA checkpoint, not a note generator --
see Section 1) as a cold-start point; change `QWEN_MODEL_PATH`/
`QWEN_LORA_ADAPTER_PATH` in Section 1 if you want a different starting model.

**Not a benchmark.** This produces one draft note per PitVis video and prints
the grounding/format scores as a sanity check, matching `run_eval.effective.sh`.
Real evaluation is human review (see COMPLIANCE.md's clinical-safety section).

## 0. Setup: Drive, paths, packages

In [1]:

import sys, subprocess, os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(r"E:/")   # adjust if running locally outside Colab

# This experiment folder itself. If you cloned/synced it to Drive elsewhere,
# point EXPERIMENT_DIR there; otherwise upload this folder's contents (at
# least `code/`) to Colab first.
EXPERIMENT_DIR = Path(os.environ.get(
    "EXPERIMENT_DIR",
    DRIVE_ROOT / "Debjyoti" / "operative-notegen-grpo-sam-countgd-qwen"
))
CODE_DIR = EXPERIMENT_DIR / "code"
assert CODE_DIR.exists(), (
    f"{CODE_DIR} not found -- upload/sync this experiment folder to Drive first, "
    "or set EXPERIMENT_DIR to wherever you placed it."
)

NOTEGEN_SRC = CODE_DIR / "models" / "qwen3vl_finetune" / "src"
RLVR_DIR = CODE_DIR / "rlvr"
sys.path.insert(0, str(NOTEGEN_SRC))
sys.path.insert(0, str(RLVR_DIR))

if IN_COLAB:
    # peft is installed here UNCONDITIONALLY (not inside Section 1a's S3-download
    # branch) because service.py's QwenNoteGenerator needs it whenever a LoRA
    # adapter is set, regardless of whether the adapter had to be downloaded this
    # run or was already cached on Drive.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "accelerate", "qwen-vl-utils[decord]",
                    "decord", "av", "pillow", "peft"], check=True)

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch: 2.11.0+cu128 | CUDA: True


In [2]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchao>=0.16.0"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], returncode=0)

## 1. Qwen model: base + LoRA adapter (A_s189)

Using `A_s189` (`s3://stanford-train-data/models/A_s189/`) as the starting
checkpoint -- confirmed 2026-09-28: another team member's **counting/VQA**
GRPO LoRA adapter (base `Qwen/Qwen3.5-9B`, rank 64/alpha 128), **not**
trained for note generation. It's used here as a cold-start point (already
surgical-domain-adapted) for this project's own GRPO run, not as a working
note generator by itself -- expect low `note_format`/`note_grounding` scores
in Section 4 below until this project's own GRPO training
(`train_grpo.effective.sh`) has actually run. See `notes.md` "Model address".

It's a **LoRA adapter** (`adapter_config.json` + `adapter_model.safetensors`
only -- no tokenizer/processor files of its own), so this downloads it
separately from the base model and merges them in Section 3
(`service.py`'s `QwenNoteGenerator(lora_adapter_path=...)`).

**Memory note (confirmed 2026-09-28, 22GB Colab GPU):** `service.py` loads
the base model + merges the LoRA adapter on **CPU**, then moves only the
merged result to GPU -- a ~9.5B bf16 model alone needs ~17.5GB, which left
too little headroom for the merge step when both happened on GPU (hit a real
CUDA OOM). The CPU merge is slower but actually fits.

### 1a. Download the A_s189 adapter from S3

Checks Drive **first** -- if `adapter_config.json` and
`adapter_model.safetensors` are already there from a previous run of this
cell, it skips S3 entirely and does not ask for AWS credentials again.
Credentials (`getpass` -- never printed or written to disk) are only
requested on an actual cache miss. Read-only: only downloads from
`stanford-train-data`, never writes/syncs there.

In [3]:

from tqdm.auto import tqdm

QWEN_LORA_LOCAL_DIR = DRIVE_ROOT / "operative-notegen" / "weights" / "A_s189"
EXPECTED_FILES = ["adapter_config.json", "adapter_model.safetensors"]

def _already_downloaded(d):
    return all((d / f).exists() and (d / f).stat().st_size > 0 for f in EXPECTED_FILES)

if _already_downloaded(QWEN_LORA_LOCAL_DIR):
    print(f"A_s189 already present at {QWEN_LORA_LOCAL_DIR} -- skipping S3 (no credential prompt).")
    keys, s3, BUCKET, S3_PREFIX = [], None, None, None   # nothing for the download cell below to do
else:
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3"], check=True)  # peft: Section 0

    import getpass
    import boto3

    AWS_ACCESS_KEY_ID = getpass.getpass("AWS access key ID: ")
    AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS secret access key: ")
    AWS_SESSION_TOKEN = getpass.getpass("AWS session token (blank if not using temporary credentials): ") or None

    session = boto3.Session(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        aws_session_token=AWS_SESSION_TOKEN,
    )
    s3 = session.client("s3")
    print("Credentials accepted for this session (not saved to disk or printed anywhere below).")

    BUCKET = "stanford-train-data"   # NOTE: different bucket than the "upload to your namespace"
                                      # one in README.md -- this is read-only shared team storage
    S3_PREFIX = "models/A_s189/"

    paginator = s3.get_paginator("list_objects_v2")
    keys = [obj["Key"] for page in paginator.paginate(Bucket=BUCKET, Prefix=S3_PREFIX)
            for obj in page.get("Contents", []) if not obj["Key"].endswith("/")]
    print(f"{len(keys)} object(s) under s3://{BUCKET}/{S3_PREFIX}")
    for k in keys:
        print(" ", k)
    if not keys:
        raise FileNotFoundError(f"Nothing found under s3://{BUCKET}/{S3_PREFIX} -- check the prefix.")


A_s189 already present at /content/drive/MyDrive/operative-notegen/weights/A_s189 -- skipping S3 (no credential prompt).


In [4]:

QWEN_LORA_LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for key in tqdm(keys, desc="Downloading A_s189"):   # empty (already-downloaded case) -> no-op loop
    rel = key[len(S3_PREFIX):]
    dest = QWEN_LORA_LOCAL_DIR / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"]:
        continue   # already downloaded, same size -- skip re-pulling it
    s3.download_file(BUCKET, key, str(dest))

QWEN_LORA_ADAPTER_PATH = str(QWEN_LORA_LOCAL_DIR)
print("A_s189 adapter available at:", QWEN_LORA_ADAPTER_PATH)


A_s189 adapter available at: /content/drive/MyDrive/operative-notegen/weights/A_s189


### 1b. Base model

`A_s189/adapter_config.json`'s own `base_model_name_or_path` is
`Qwen/Qwen3.5-9B` -- a public HF repo id, fetched directly (not from S3).
Skip 1a and leave `QWEN_LORA_ADAPTER_PATH = None` here if you'd rather run
the bare base model with no adapter at all.

In [5]:

QWEN_MODEL_PATH = "Qwen/Qwen3.5-9B"   # base model A_s189 was trained against
QWEN_LORA_ADAPTER_PATH = QWEN_LORA_ADAPTER_PATH if "QWEN_LORA_ADAPTER_PATH" in dir() else None
print("Base model:", QWEN_MODEL_PATH, "| LoRA adapter:", QWEN_LORA_ADAPTER_PATH or "(none -- bare base model)")


Base model: Qwen/Qwen3.5-9B | LoRA adapter: /content/drive/MyDrive/operative-notegen/weights/A_s189


## 2. PitVis video + taxonomy

Reuses whatever PitVis-2023 data you already have staged (this session's
endofm-benchmark project has it at `E:/endofm-benchmark/pitvis`, or
`/content/drive/MyDrive/endofm-benchmark/data/pitvis` if you ran the Colab
version of `benchmark_endofm_pitvis.ipynb`). Adjust the paths below to
wherever yours actually are.

In [6]:

from notegen.taxonomy import load_pitvis_taxonomy
from notegen.facts import facts_from_labels

PITVIS_VIDEOS_DIR = DRIVE_ROOT / "endofm-benchmark" / "data" / "pitvis" / "raw" / "videos"
PITVIS_ANNOT_DIR = DRIVE_ROOT / "endofm-benchmark" / "data" / "pitvis" / "raw" / "annotations"

tax = load_pitvis_taxonomy(PITVIS_ANNOT_DIR)
print(f"steps: {len(tax.steps)} | instruments: {len(tax.instruments)}")
if not tax.steps:
    print("WARNING: no map_steps.csv found -- point PITVIS_ANNOT_DIR at the right place, "
          "or the grounding reward below will have nothing to check claims against.")

VIDEO_ID = "01"   # which PitVis video to demo on
annot_csv = PITVIS_ANNOT_DIR / f"annotations_{VIDEO_ID}.csv"
video_path = PITVIS_VIDEOS_DIR / f"video_{VIDEO_ID}.mp4"
assert annot_csv.exists() and video_path.exists(), (video_path, annot_csv)

video_facts = facts_from_labels(annot_csv, tax, VIDEO_ID)
print(f"{len(video_facts.phases)} phase(s) | instruments seen: {sorted(video_facts.all_instruments_present)}")


steps: 15 | instruments: 21
78 phase(s) | instruments seen: ['bipolar_forceps', 'cottle', 'cup_forceps', 'dural_scissors', 'freer_elevator', 'haemostatic_foam', 'irrigation_syringe', 'kerrisons', 'micro_doppler', 'nasal_cutting_forceps', 'no_secondary_instrument', 'occluded_image_inside_patient', 'out_of_patient', 'pituitary_rongeurs', 'retractable_knife', 'ring_curette', 'spatula_dissector', 'stealth_pointer', 'suction', 'tissue_glue']


## 3. Generate the draft note

In [7]:

from notegen.service import QwenNoteGenerator, generate_note

MAX_PHASES = 5   # small sample for the demo -- see markdown above; None = every phase

generator = QwenNoteGenerator(model_path=QWEN_MODEL_PATH,
                               lora_adapter_path=QWEN_LORA_ADAPTER_PATH, device=DEVICE)
result = generate_note(video_path, video_facts, generator, max_phases=MAX_PHASES,
                        procedure_hint="endoscopic transsphenoidal pituitary surgery")

print(result.assembled_note)


Generating per-phase notes:   0%|          | 0/5 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Procedure narrative (5 phase(s), instruments observed: cottle, freer_elevator, kerrisons, no_secondary_instrument, occluded_image_inside_patient, out_of_patient, pituitary_rongeurs, spatula_dissector, suction)

## out_of_patient (0s-2s)
<procedure>...</procedure>
<findings>...</findings>
<steps>...</steps>
<instruments>...</instruments>
<complications>...</complications>
<closure>...</closure>

## nasal corridor creation (3s-173s)
<procedure>Endoscopic transsphenoidal pituitary surgery.</procedure>
<findings>Narrow nasal corridor, occluded by specimen bag and external drain.</findings>
<steps>1. Remove specimen bag.
2. Remove external drain.</steps>
<instruments>Specimen bag, external drain.</instruments>
<complications>None.</complications>
<closure>None required.</closure>

## anterior sphenoidotomy (174s-377s)
<procedure>Endoscopic transsphenoidal pituitary surgery.</procedure>
<findings>Anterior sphenoidotomy performed.</findings>
<steps>Cottle maneuver applied, Kerrison rongeurs u

## 4. Grounding / format sanity check

Same ORMs `train_grpo.effective.sh` would use as the GRPO reward, run
standalone here as a smoke check (not a benchmark -- see notes.md).

In [8]:

from notegen_plugin import NoteFormatORM, NoteGroundingORM

fmt, grounding = NoteFormatORM(), NoteGroundingORM()
gi = ", ".join(sorted(video_facts.all_instruments_present))
gs = ", ".join(sorted(video_facts.all_steps_present))

fmt_scores = fmt(result.per_phase_raw)
ground_scores = grounding(result.per_phase_raw,
                           facts_instruments=[gi] * len(result.per_phase_raw),
                           facts_steps=[gs] * len(result.per_phase_raw))

for i, (f, g) in enumerate(zip(fmt_scores, ground_scores)):
    print(f"phase {i} ({video_facts.phases[i].step_name}): note_format={f:.2f}  note_grounding={g:.2f}")
print(f"\nmean note_format={sum(fmt_scores)/len(fmt_scores):.2f}  "
      f"mean note_grounding={sum(ground_scores)/len(ground_scores):.2f}")
print("\nLow note_format -> the model isn't following the <facts_used>/<note> template yet; "
      "consider a few-shot example in notegen/prompts.py's SYSTEM_PROMPT before running GRPO.")
print("Low note_grounding -> the model is claiming instruments/steps the facts don't support "
      "(or under-mentioning ones that ARE present) -- exactly what the GRPO reward is meant to fix.")


phase 0 (out_of_patient): note_format=1.00  note_grounding=0.00
phase 1 (nasal corridor creation): note_format=1.00  note_grounding=0.00
phase 2 (anterior sphenoidotomy): note_format=1.00  note_grounding=0.30
phase 3 (out_of_patient): note_format=0.00  note_grounding=0.00
phase 4 (anterior sphenoidotomy): note_format=1.00  note_grounding=0.20

mean note_format=0.80  mean note_grounding=0.10

Low note_format -> the model isn't following the <facts_used>/<note> template yet; consider a few-shot example in notegen/prompts.py's SYSTEM_PROMPT before running GRPO.
Low note_grounding -> the model is claiming instruments/steps the facts don't support (or under-mentioning ones that ARE present) -- exactly what the GRPO reward is meant to fix.


## Next steps (not done by this notebook)

1. Run this over more PitVis videos (loop `VIDEO_ID`) to see how format/grounding
   vary before spending GRPO compute.
2. Build the GRPO training `.jsonl` (one row per phase, across PitVis videos,
   with `facts_instruments`/`facts_steps` columns -- see
   `code/rlvr/notegen_plugin.py`'s docstring for the exact shape).
3. Run `train_grpo.effective.sh` with `QWEN_MODEL_PATH`, `QWEN_MODEL_TYPE`, and
   `NOTEGEN_TRAIN_JSONL` set.
4. Re-run this notebook against the fine-tuned checkpoint and compare
   format/grounding scores -- still not a substitute for human review before
   trusting note content (see COMPLIANCE.md).